
### Web-Based Text Summarization Using GenAI 

In the era of information overload, extracting meaningful insights from extensive textual content is a critical need. 
This project focuses on developing a **web-based text summarization tool**, leveraging Natural Language Processing (NLP) techniques 
to generate concise summaries from long-form content. The tool is designed to provide accurate and context-aware summarization 
to aid users in consuming key information efficiently.

In [1]:
import os
import requests
from dotenv import load_dotenv
from bs4 import BeautifulSoup
import openai
from langchain_openai import ChatOpenAI
load_dotenv()
os.environ['OPENAI_API_KEY'] = os.getenv("OPENAI_API_KEY")
## Langsmith Tracking
os.environ["LANGCHAIN_API_KEY"]=os.getenv("LANGCHAIN_API_KEY")
os.environ["LANGCHAIN_TRACING_V2"]="true"
os.environ["LANGCHAIN_PROJECT"]=os.getenv("LANGCHAIN_PROJECT")

In [2]:
client = openai.OpenAI()  
response = client.chat.completions.create(
    model="gpt-4o",
    messages=[{"role": "user", "content": "Your message here"}]
)
print(response.choices[0].message.content)

Hello! How can I assist you today?


In [3]:
# A class to represent a Webpage
# Some websites need you to use proper headers when fetching them:
headers = {
 "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/117.0.0.0 Safari/537.36"
}
class Website:
    def __init__(self, url):
        """
        Create this Website object from the given url using the BeautifulSoup library
        """
        self.url = url
        response = requests.get(url, headers=headers)
        soup = BeautifulSoup(response.content, 'html.parser')
        self.title = soup.title.string if soup.title else "No title found"
        for irrelevant in soup.body(["script", "style", "img", "input"]):
            irrelevant.decompose()
        self.text = soup.body.get_text(separator="\n", strip=True)

In [4]:
# Let's try one out. Change the website and add print statements to follow along.

ed = Website("https://en.wikipedia.org/wiki/Ed_Sheeran")
print(ed.title)
print(ed.text)

Ed Sheeran - Wikipedia
Jump to content
Main menu
Main menu
move to sidebar
hide
Navigation
Main page
Contents
Current events
Random article
About Wikipedia
Contact us
Contribute
Help
Learn to edit
Community portal
Recent changes
Upload file
Search
Search
Appearance
Donate
Create account
Log in
Personal tools
Donate
Create account
Log in
Pages for logged out editors
learn more
Contributions
Talk
Contents
move to sidebar
hide
(Top)
1
Early life and education
2
Career
Toggle Career subsection
2.1
2004–2010: Career beginnings
2.2
2011–2013:
+
("
Plus
")
2.3
2014–2015:
×
("
Multiply
")
2.4
2016–2018:
÷
("
Divide
")
2.5
2019–2022:
No.6 Collaborations Project
and
=
("
Equals
")
2.6
2023–present:
-
("
Subtract
") and
Autumn Variations
3
Musical style and influences
4
Other ventures
Toggle Other ventures subsection
4.1
Gingerbread Man Records
4.2
Bertie Blossoms
4.3
Charity work
4.4
Acting
5
Impact
6
Accolades
7
Personal life
8
Legal issues
9
Political views
10
Discography
11
Filmography
Toggle

In [5]:
# Define our system prompt - you can experiment with this later, changing the last sentence to 'Respond in markdown in Spanish."

system_prompt = "You are an assistant that analyzes the contents of a website \
                and provides a short summary, ignoring text that might be navigation related. \
                Respond in markdown."

In [6]:
# A function that writes a User Prompt that asks for summaries of websites:

def user_prompt_for(website):
    user_prompt = f"You are looking at a website titled {website.title}\n"
    user_prompt += "The contents of this website is as follows; please provide a short summary of this website in markdown.\n"
    user_prompt += "If it includes news or announcements, then summarize these too.\n\n"
    #user_prompt += website.text
    return user_prompt

In [7]:
print(user_prompt_for(ed))

You are looking at a website titled Ed Sheeran - Wikipedia
The contents of this website is as follows; please provide a short summary of this website in markdown.
If it includes news or announcements, then summarize these too.




In [8]:
# See how this function creates exactly the format above
def messages_for(website):
    return [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt_for(website)}
    ]

In [9]:
# Try this out, and then try for a few more websites
print(messages_for(ed))

[{'role': 'system', 'content': 'You are an assistant that analyzes the contents of a website                 and provides a short summary, ignoring text that might be navigation related.                 Respond in markdown.'}, {'role': 'user', 'content': 'You are looking at a website titled Ed Sheeran - Wikipedia\nThe contents of this website is as follows; please provide a short summary of this website in markdown.\nIf it includes news or announcements, then summarize these too.\n\n'}]


In [10]:
# And now: call the OpenAI API. You will get very familiar with this!
def summarize(url):
    website = Website(url)
    response = openai.chat.completions.create(
        model = "gpt-4o-mini",
        messages = messages_for(website)
    )
    return response.choices[0].message.content

In [11]:
summarize("https://en.wikipedia.org/wiki/Ed_Sheeran")

'# Ed Sheeran - Wikipedia Summary\n\nThe Wikipedia page for Ed Sheeran provides a comprehensive overview of the English singer-songwriter\'s life and career. It details his early life, musical influences, and how he rose to fame, including notable albums and hit singles. \n\n## Career Highlights\n- **Early Work**: Information about Ed\'s initial efforts in music, including his first EPs and performances.\n- **Breakthrough Albums**: Analysis of his major albums like "+" (Plus), "x" (Multiply), and "÷" (Divide), showcasing their critical and commercial success.\n- **Collaborations**: A list of notable collaborations with other artists, highlighting his versatility and popularity in the music industry.\n\n## Notable Achievements\n- **Awards**: Overview of various accolades and nominations he has received throughout his career.\n- **Impact**: Discussion on his influence in the contemporary music scene, particularly in pop and folk genres.\n\n## Recent News\nAs of the last update, informati

In [12]:
# A function to display this nicely in the Jupyter output, using markdown

def display_summary(url):
    summary = summarize(url)
    display(summary)

In [13]:
display_summary("https://en.wikipedia.org/wiki/Ed_Sheeran")

'# Ed Sheeran - Wikipedia Summary\n\nThis webpage provides a comprehensive overview of Ed Sheeran, a renowned English singer-songwriter. It covers his early life, musical career, and rise to fame, detailing key albums, hits, and collaborations. Ed\'s distinctive style combines elements of pop, folk, and hip-hop, which has contributed to his widespread popularity.\n\n### Notable Highlights:\n- **Early Life**: Information about his childhood, influences, and beginnings in music.\n- **Career Milestones**: An account of major albums like "+", "x", "÷", and "=", including notable singles like "Shape of You" and "Thinking Out Loud".\n- **Awards & Achievements**: A summary of awards received throughout his career, showcasing his impact on the music industry.\n- **Collaborations**: Notable partnerships with other artists, which highlight his versatility and reach in the music world.\n\nCurrently, the site does not mention any recent news or announcements regarding Ed Sheeran.'

In [14]:
display_summary("https://cnn.com")

'# Summary of CNN: Breaking News, Latest News, and Videos\n\nThe CNN website is a comprehensive source for breaking news and the latest developments across various topics including politics, health, technology, and entertainment. It offers timely articles, opinion pieces, and extensive multimedia content such as videos and photo galleries. \n\n### Key Features:\n- **Breaking News:** Up-to-the-minute news updates on significant events happening globally.\n- **In-Depth Reports:** Detailed articles providing analysis and context on major news stories.\n- **Multimedia Content:** A variety of videos and images that complement news articles and enhance user engagement.\n- **Interactive Elements:** User comments and social media integration for discussion and engagement.\n\n### Recent News Highlights:\n- Major political developments at home and abroad.\n- Updates on ongoing global health issues and policy changes.\n- Coverage of significant cultural events, including entertainment news.\n\nOv

In [ ]:
display_summary("https://anthropic.com")